# Day 19 — Mocks

> ⚠️ **Why this matters.** Your `api.py` hits the real internet. You don't want tests that:
> - Slow down because of network
> - Fail when the internet is out
> - Cost real API rate-limit budget
> - Behave differently in CI
> 
> Mocks let you say: 'when the code calls `fetch_word("thorough")`, return this fake response instead'. Same test logic, no network.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/19-mocks.ipynb)

## What you'll do today

- [ ] You know what `Mock` and `MagicMock` are
- [ ] You can patch a function in another module with `monkeypatch`
- [ ] You can use `unittest.mock.patch` (decorator or context manager)
- [ ] Your `api.py` is tested without network calls

## 1. The simplest mock — `monkeypatch`

In [ ]:
# api.py
import requests

def fetch_word(word):
    r = requests.get(f'https://api.dictionaryapi.dev/api/v2/entries/en/{word}', timeout=5)
    if r.status_code == 404: return None
    return r.json()[0]

In [ ]:
# tests/test_api.py
from english_helper import api

def test_fetch_word_returns_none_on_404(monkeypatch):
    class FakeResponse:
        status_code = 404
        def json(self): return []
    
    def fake_get(url, timeout):
        return FakeResponse()
    
    monkeypatch.setattr(api.requests, 'get', fake_get)
    
    result = api.fetch_word('nope')
    assert result is None

**What's happening:**

1. We define a fake response class with the attributes our code accesses.
2. We define a fake `get` function returning it.
3. `monkeypatch.setattr` replaces `api.requests.get` with our fake — only for this test.
4. The test calls `api.fetch_word(...)`, which calls (the now-fake) `requests.get`.

After the test: pytest auto-restores `requests.get`. No leakage.

## 2. `MagicMock` — the lazy way

In [ ]:
from unittest.mock import MagicMock

def test_fetch_word_returns_first_entry(monkeypatch):
    fake = MagicMock()
    fake.status_code = 200
    fake.json.return_value = [{'word': 'thorough', 'phonetic': '/x/'}]
    
    monkeypatch.setattr(api.requests, 'get', MagicMock(return_value=fake))
    
    result = api.fetch_word('thorough')
    assert result['word'] == 'thorough'

`MagicMock` is like a chameleon — it acquires whatever attribute or method you access on it. `.json` becomes a callable that returns whatever you set as `return_value`.

**Bonus:** you can assert how it was called:

In [ ]:
def test_fetch_word_calls_correct_url(monkeypatch):
    fake_get = MagicMock()
    fake_get.return_value.status_code = 404
    monkeypatch.setattr(api.requests, 'get', fake_get)
    
    api.fetch_word('thorough')
    
    # Assert the call
    fake_get.assert_called_once_with(
        'https://api.dictionaryapi.dev/api/v2/entries/en/thorough',
        timeout=5
    )

**Useful assertions:**

- `mock.assert_called_once()` — exactly one call
- `mock.assert_called_with(...)` — last call args
- `mock.assert_called_once_with(...)` — exactly one call with these args
- `mock.call_count` — how many calls
- `mock.call_args_list` — every call's args

## 3. `unittest.mock.patch` decorator

In [ ]:
from unittest.mock import patch

@patch('english_helper.api.requests.get')
def test_fetch_word(mock_get):
    mock_get.return_value.status_code = 200
    mock_get.return_value.json.return_value = [{'word': 'thorough'}]
    
    result = api.fetch_word('thorough')
    assert result['word'] == 'thorough'

**The path you patch matters.** Always patch where the function is *used*, not where it's defined. `api.py` does `import requests`; tests patch `english_helper.api.requests.get` (the reference in api.py), not `requests.get` (the original).

> 💡 **Tip:** if a patch "doesn't work" (real function still runs), check the patch path.

## 4. When to mock vs not

**Mock:**
- Network calls (always)
- Real file system writes (when slow or hard to set up — `tmp_path` is usually enough though)
- Time (`datetime.now()`) for deterministic tests
- Random (`random.choice`) for deterministic tests
- Anything outside the unit you're testing

**Don't mock:**
- Pure functions inside your codebase
- Dataclasses
- Logic you're trying to test

**Sign of trouble: you mock 10 things in one test.** That's a smell — the function does too much. Refactor.

## End-of-day mini-project — mock the API

> 🎯 **Add `tests/test_api.py` with mocked tests.**

### Required tests

- `test_fetch_word_returns_entry_on_200`
- `test_fetch_word_returns_none_on_404`
- `test_fetch_word_returns_none_on_timeout` (patch `requests.get` to raise `requests.Timeout`)
- `test_fetch_word_correct_url` — assert called with correct URL
- `test_extract_ipa_from_real_response` (use a fixture with realistic data, no mock needed — it's a pure function)

All tests pass in <1 second. **Zero network calls.**

## Connect to the project

> 🎯 **Tomorrow (Day 20):** coverage — measure what your tests cover. End of Week 4.

**Quiz:** [19-mocks-quiz.ipynb](19-mocks-quiz.ipynb)